# Phase 6a — Merge LoRA + Export GGUF (q4_k_m) + Push to HF Hub

One-shot session, ~30–45 min (GGUF conversion compiles llama.cpp — the long
quiet stretch is normal). **Before running:** Kaggle → Add-ons → Secrets →
add secret named `HF_TOKEN` with a *write* token from hf.co/settings/tokens.

In [ ]:
# CELL 1 — config + secret
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
ADAPTER_REPO = "this-is-rickroy/phi3-mini-spider-qlora-r16"
GGUF_REPO    = "this-is-rickroy/phi3-mini-spider-sql-gguf"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# CELL 2 — pinned install
%pip install -q unsloth==2026.6.9 trl==0.24.0 peft==0.19.1 transformers==5.5.0

In [ ]:
# CELL 3 — load adapter, merge, export GGUF q4_k_m (the slow cell)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_REPO, max_seq_length=3584, dtype=None, load_in_4bit=True)
model.save_pretrained_gguf("gguf_out", tokenizer, quantization_method="q4_k_m")
!ls -la gguf_out/

In [ ]:
# CELL 4 — push to Hub
from huggingface_hub import HfApi
import glob, os
api = HfApi(token=os.environ["HF_TOKEN"])
api.create_repo(GGUF_REPO, repo_type="model", exist_ok=True)
gguf_file = glob.glob("gguf_out/*q4_k_m*.gguf")[0]
print("uploading", gguf_file)
api.upload_file(path_or_fileobj=gguf_file,
                path_in_repo="phi3-mini-spider-sql.q4_k_m.gguf",
                repo_id=GGUF_REPO)
print(f"done -> https://huggingface.co/{GGUF_REPO}")